# OPV Bayesian Optimization Recommendation Audit

This notebook demonstrates how to audit finite-pool Bayesian-optimization recommendations with `matgpr`. It uses the OPV descriptor dataset as a retrospective materials-informatics example: a small subset is treated as already measured, the remaining rows are treated as candidate materials, and the workflow ranks the next OPV candidates to evaluate.

The important output is not only the ranked list. The audit tables explain why each candidate was recommended: acquisition score, posterior uncertainty, feasibility, trust-region status, duplicate status, and batch-selection order.

If the optional BoTorch dependency is available, the notebook runs a real GPR-based acquisition ranking. If BoTorch is not installed, it falls back to a deterministic ranked-table example so the audit workflow remains executable.

## 1. Setup

The cache environment variables are set before importing `matgpr` because the package exposes plotting helpers that import Matplotlib.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

cache_root = Path(tempfile.gettempdir()) / "matgpr_notebook_cache"
os.environ.setdefault("MPLCONFIGDIR", str(cache_root / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(cache_root / "xdg"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise RuntimeError("Could not find the matgpr project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from matgpr import (
    CandidateConstraint,
    CandidateDuplicatePolicy,
    CandidateTrustRegion,
    apply_candidate_constraints,
    apply_candidate_duplicate_policy,
    apply_candidate_trust_region,
    fit_botorch_surrogate,
    is_optional_dependency_available,
    rank_discrete_candidates,
    select_diverse_batch,
    summarize_bo_recommendation_audit,
    summarize_candidate_pool,
)

RANDOM_STATE = 42
MEASURED_COUNT = 32
CANDIDATE_COUNT = 96
TOP_K = 5
BO_FIT_MODEL = True
TARGET_COLUMN = "PCE"

BASE_FEATURE_COLUMNS = [
    "polarizability",
    "delLA",
    "delLD",
    "N_atom",
    "Eg",
    "lamda_h",
    "DIP",
    "AL-DH",
    "delHD",
    "E_bind",
    "DL-AL",
    "delGE",
    "E_T1",
]
PHYSICS_COLUMNS = ["physics_degeneracy_score", "physics_binding_score"]
DIVERSITY_COLUMNS = ["Eg", "E_bind", "delHD", "delLD", "delLA", *PHYSICS_COLUMNS]

plt.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 2. Load The OPV Candidate Pool

For a real campaign, the measured table would contain completed experiments and the candidate table would contain unmeasured materials. Here the target values are known because this is a retrospective example. The candidate target is renamed to `withheld_pce_for_retrospective` and is used only for post-ranking inspection, not for acquisition ranking.

In [ ]:
def load_opv_data() -> pd.DataFrame:
    data_path = PROJECT_ROOT / "examples" / "opv" / "dataset.pkl"
    data = pd.read_pickle(data_path)
    data = data.rename(columns={"#Sno.": "candidate_id"})
    data["candidate_id"] = "opv_" + data["candidate_id"].astype(str)
    keep_columns = ["candidate_id", TARGET_COLUMN, *BASE_FEATURE_COLUMNS]
    return data.loc[:, keep_columns].dropna().reset_index(drop=True)


opv_data = load_opv_data()
measured_data = opv_data.sample(n=MEASURED_COUNT, random_state=RANDOM_STATE).reset_index(drop=True)
candidate_pool = opv_data.loc[~opv_data["candidate_id"].isin(measured_data["candidate_id"])]
candidate_pool = candidate_pool.sample(n=CANDIDATE_COUNT, random_state=RANDOM_STATE + 1).reset_index(drop=True)

print(f"Measured training rows: {len(measured_data)}")
print(f"Candidate rows: {len(candidate_pool)}")
opv_data.head()

## 3. Build BO Features And Metadata

The BO surrogate uses OPV descriptors plus two compact physics scores:

- `physics_degeneracy_score`: favorable when donor/acceptor frontier-orbital gaps are small.
- `physics_binding_score`: favorable when exciton binding energy is low.

The z-score reference statistics are computed from the measured training rows only.

In [ ]:
def add_physics_scores(frame: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    result = frame.loc[:, BASE_FEATURE_COLUMNS].astype(float).copy()
    physics_columns = ["delHD", "delLD", "delLA", "E_bind"]
    physics_source = frame.loc[:, physics_columns].astype(float)
    reference_physics = reference.loc[:, physics_columns].astype(float)
    reference_mean = reference_physics.mean(axis=0)
    reference_std = reference_physics.std(axis=0, ddof=0).replace(0.0, 1.0)
    z_scores = (physics_source - reference_mean) / reference_std
    result["physics_degeneracy_score"] = -(
        z_scores["delHD"] + z_scores["delLD"] + z_scores["delLA"]
    ) / 3.0
    result["physics_binding_score"] = -z_scores["E_bind"]
    return result


X_train_raw = add_physics_scores(measured_data, measured_data)
X_candidates_raw = add_physics_scores(candidate_pool, measured_data)

scaler = StandardScaler().fit(X_train_raw)
X_train = pd.DataFrame(scaler.transform(X_train_raw), columns=X_train_raw.columns)
X_candidates = pd.DataFrame(scaler.transform(X_candidates_raw), columns=X_candidates_raw.columns)

candidate_metadata = candidate_pool.loc[
    :, ["candidate_id", TARGET_COLUMN, "Eg", "E_bind", "delHD", "delLD", "delLA"]
].rename(columns={TARGET_COLUMN: "withheld_pce_for_retrospective"})
candidate_metadata = pd.concat(
    [
        candidate_metadata.reset_index(drop=True),
        X_candidates_raw.loc[:, PHYSICS_COLUMNS].reset_index(drop=True),
    ],
    axis=1,
)

candidate_metadata.head()

## 4. Audit The Candidate Pool Before BO

Candidate-pool diagnostics are separate from BO. They check whether identifiers are duplicated and whether numeric descriptors are complete before any acquisition function is evaluated.

In [ ]:
pool_diagnostics = summarize_candidate_pool(
    candidate_metadata,
    feature_columns=DIVERSITY_COLUMNS,
    key_columns=("candidate_id",),
)

display(pool_diagnostics.overview_frame())
display(pool_diagnostics.numeric_feature_frame().head(10))

## 5. Define Feasibility, Trust-Region, And Duplicate Policies

These policies are intentionally simple and demonstrative. In a real campaign, they should come from synthesis constraints, processing windows, instrument limits, safety rules, or project-specific chemistry knowledge.

In [ ]:
constraints = [
    CandidateConstraint(
        name="moderate_bandgap_window",
        column="Eg",
        lower_bound=380.0,
        upper_bound=560.0,
    ),
    CandidateConstraint(
        name="binding_energy_limit",
        column="E_bind",
        upper_bound=2.7,
    ),
]

center = X_train.mean(axis=0).to_frame().T
candidate_distances = np.linalg.norm(
    X_candidates.to_numpy(dtype=float) - center.to_numpy(dtype=float),
    axis=1,
)
trust_region = CandidateTrustRegion(
    centers=center,
    radius=float(np.quantile(candidate_distances, 0.80)),
)

pending_candidate = candidate_metadata.sample(n=1, random_state=RANDOM_STATE + 2)[["candidate_id"]]
duplicate_policy = CandidateDuplicatePolicy(
    existing_candidates=pending_candidate,
    key_columns=("candidate_id",),
)

print(f"Trust-region radius: {trust_region.radius:.3f}")
print("Simulated pending candidate:")
display(pending_candidate)

## 6. Rank Candidates

When BoTorch is available, the notebook fits a small exact GP surrogate on the measured rows and ranks the finite candidate pool with upper confidence bound. Policy columns are annotated rather than filtered so the audit summary can report how many candidates fail each rule.

The fallback path is clearly labeled and exists only so this notebook can run without the optional `matgpr[bo]` extra.

In [ ]:
def deterministic_ranked_table(
    candidate_features: pd.DataFrame,
    metadata: pd.DataFrame,
    constraints: list[CandidateConstraint],
    trust_region: CandidateTrustRegion,
    duplicate_policy: CandidateDuplicatePolicy,
) -> pd.DataFrame:
    """Create a deterministic ranked table when BoTorch is unavailable."""
    table = metadata.copy()
    feature_radius = np.linalg.norm(candidate_features.to_numpy(dtype=float), axis=1)
    uncertainty = 0.15 + 0.25 * (feature_radius / max(float(feature_radius.max()), 1.0))
    predicted_mean = (
        3.0
        + 0.55 * metadata["physics_degeneracy_score"].to_numpy(dtype=float)
        + 0.35 * metadata["physics_binding_score"].to_numpy(dtype=float)
        - 0.20 * metadata["E_bind"].to_numpy(dtype=float)
    )
    table["matgpr_predicted_mean"] = predicted_mean
    table["matgpr_predicted_std"] = uncertainty
    table["matgpr_acquisition"] = predicted_mean + 0.5 * uncertainty
    table = apply_candidate_constraints(table, constraints)
    table = apply_candidate_trust_region(
        table,
        trust_region,
        candidate_features=candidate_features,
        policy="annotate",
    )
    table = apply_candidate_duplicate_policy(
        table,
        duplicate_policy,
        policy="annotate",
    )
    table = table.sort_values("matgpr_acquisition", ascending=False).reset_index(drop=True)
    table.insert(0, "matgpr_rank", np.arange(1, len(table) + 1))
    return table


if is_optional_dependency_available("botorch"):
    surrogate = fit_botorch_surrogate(
        X_train,
        measured_data[TARGET_COLUMN],
        maximize=True,
        normalize_features=True,
        standardize_target=True,
        fit_model=BO_FIT_MODEL,
    )
    ranked_candidates = rank_discrete_candidates(
        surrogate,
        X_candidates,
        acquisition_function="upper_confidence_bound",
        beta=0.4,
        candidate_data=candidate_metadata,
        constraints=constraints,
        constraint_policy="annotate",
        trust_region=trust_region,
        trust_region_policy="annotate",
        duplicate_policy=duplicate_policy,
        duplicate_policy_action="annotate",
    )
    ranking_source = "BoTorch upper confidence bound"
else:
    ranked_candidates = deterministic_ranked_table(
        X_candidates,
        candidate_metadata,
        constraints,
        trust_region,
        duplicate_policy,
    )
    ranking_source = "deterministic fallback ranked table"

print(f"Ranking source: {ranking_source}")
ranked_candidates.head(10)

## 7. Select An Eligible Diverse Batch

The complete ranked table keeps every candidate and every policy annotation. The recommended batch is then selected only from candidates that are feasible, inside the trust region, and not duplicates. This preserves the full audit trail while keeping the actual recommendations experimentally realistic.

In [ ]:
def add_batch_annotations(
    ranked_candidates: pd.DataFrame,
    selected_batch: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    result = ranked_candidates.copy()
    result["matgpr_batch_selected"] = False
    result["matgpr_batch_order"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result["matgpr_batch_score"] = np.nan

    if selected_batch.empty:
        return result, selected_batch

    order_map = selected_batch.set_index("candidate_id")["matgpr_batch_order"].to_dict()
    score_map = selected_batch.set_index("candidate_id")["matgpr_batch_score"].to_dict()
    for candidate_id, order in order_map.items():
        mask = result["candidate_id"] == candidate_id
        result.loc[mask, "matgpr_batch_selected"] = True
        result.loc[mask, "matgpr_batch_order"] = int(order)
        result.loc[mask, "matgpr_batch_score"] = float(score_map[candidate_id])

    recommendations = (
        result.loc[result["matgpr_batch_selected"]]
        .sort_values("matgpr_batch_order")
        .reset_index(drop=True)
    )
    return result, recommendations


eligible_mask = (
    ranked_candidates["matgpr_feasible"].astype(bool)
    & ranked_candidates["matgpr_in_trust_region"].astype(bool)
    & ~ranked_candidates["matgpr_is_duplicate"].astype(bool)
)
eligible_candidates = ranked_candidates.loc[eligible_mask].reset_index(drop=True)

selected_batch = select_diverse_batch(
    eligible_candidates,
    top_k=min(TOP_K, len(eligible_candidates)),
    score_column="matgpr_acquisition",
    feature_columns=DIVERSITY_COLUMNS,
    diversity_weight=0.2,
    return_all=False,
)
ranked_with_batch, recommendations = add_batch_annotations(ranked_candidates, selected_batch)

print(f"Eligible candidates: {len(eligible_candidates)} / {len(ranked_candidates)}")
display(
    recommendations.loc[
        :,
        [
            "candidate_id",
            "matgpr_rank",
            "matgpr_acquisition",
            "matgpr_predicted_mean",
            "matgpr_predicted_std",
            "matgpr_batch_order",
            "withheld_pce_for_retrospective",
        ],
    ]
)

## 8. Summarize Recommendation Audit Tables

`BORecommendationAudit` returns four report-ready tables:

- `overview_frame()`: one-row summary of the campaign decision.
- `policy_summary_frame()`: feasibility, trust-region, duplicate, and batch-selection counts.
- `score_summary_frame()`: recommended-versus-ranked score ranges.
- `recommendation_frame()`: per-candidate audit notes.

In [ ]:
recommendation_audit = summarize_bo_recommendation_audit(
    recommendations,
    ranked_candidates=ranked_with_batch,
    candidate_count=len(candidate_pool),
    identifier_columns=("candidate_id",),
)

display(recommendation_audit.overview_frame())
display(recommendation_audit.policy_summary_frame())
display(recommendation_audit.score_summary_frame())
display(recommendation_audit.recommendation_frame())

## 9. Visual Check

This retrospective plot compares predicted PCE and withheld experimental PCE for the ranked pool. The withheld target is shown only because the OPV dataset is historical; it would not be available for truly unmeasured candidates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

plot_data = ranked_with_batch.copy()
selected_mask = plot_data["matgpr_batch_selected"].astype(bool)

axes[0].scatter(
    plot_data["matgpr_predicted_std"],
    plot_data["matgpr_acquisition"],
    s=24,
    color="#8a8f98",
    alpha=0.65,
    label="ranked candidates",
)
axes[0].scatter(
    plot_data.loc[selected_mask, "matgpr_predicted_std"],
    plot_data.loc[selected_mask, "matgpr_acquisition"],
    s=60,
    color="#007c89",
    edgecolor="black",
    linewidth=0.5,
    label="recommended batch",
)
axes[0].set_xlabel("Predicted standard deviation")
axes[0].set_ylabel("Acquisition score")
axes[0].set_title("Acquisition versus uncertainty")
axes[0].legend(frameon=False)

axes[1].errorbar(
    recommendations["withheld_pce_for_retrospective"],
    recommendations["matgpr_predicted_mean"],
    yerr=recommendations["matgpr_predicted_std"],
    fmt="o",
    color="#007c89",
    ecolor="#007c89",
    elinewidth=1.0,
    capsize=3,
)
min_value = min(
    recommendations["withheld_pce_for_retrospective"].min(),
    recommendations["matgpr_predicted_mean"].min(),
)
max_value = max(
    recommendations["withheld_pce_for_retrospective"].max(),
    recommendations["matgpr_predicted_mean"].max(),
)
axes[1].plot([min_value, max_value], [min_value, max_value], color="#444444", linestyle="--")
axes[1].set_xlabel("Withheld experimental PCE (%)")
axes[1].set_ylabel("Predicted PCE (%)")
axes[1].set_title("Recommended candidates")

plt.show()

## 10. Takeaways

The audit pattern makes BO recommendations easier to defend. A materials scientist can see whether a candidate was selected because of a high predicted value, high uncertainty, diversity within the batch, or policy eligibility. The same pattern can be reused for composition grids, polymer libraries, formulation spaces, or processing-condition candidates.